# Research Agent evaluation: content analyzer

Evaluates `_analyze_single_content` (the LLM that describes one Instagram post)
with LLM-as-a-judge in LangSmith.

- **Data:** 20 Dear Duck posts from `data/dearduck_sa_research_20260922_112749.json`.
- **Images:** saved in `data/images/`. The Instagram links in the JSON expired on
  2026-09-26, so the analyzer and the judges both use these saved copies, which
  means they always see the same images.
- **Evaluators (one score each):**
  - `grounding` (LLM judge): claims, summaries and evidence are supported by the post.
  - `visual_accuracy` (LLM judge): visual signals, on-image text and visual summary match the images.
  - `interpretation` (LLM judge): pillar, categories, themes, CTA and promotion are reasonable.
  - `neutrality` (code, free): the analysis describes and does not judge (Research Agent role).

Version : 1

## 1. Setup

In [1]:
# Install once if needed (no -U, so the tested package versions are kept):
# %pip install langsmith openevals python-dotenv

In [2]:
import base64
import json
import sys
from pathlib import Path

from dotenv import load_dotenv
from langsmith import Client
from openevals.llm import create_llm_as_judge

project_root = next(
    (
        path
        for path in [Path.cwd(), *Path.cwd().parents]
        if (path / "agents" / "research_agent" / "research_analyzer.py").is_file()
    ),
    None,
)
if project_root is None:
    raise FileNotFoundError("Could not locate the project root.")
sys.path.insert(0, str(project_root))

load_dotenv(project_root / ".env")
print("LangSmith key:", bool(__import__("os").getenv("LANGSMITH_API_KEY")))
print("OpenAI key:", bool(__import__("os").getenv("OPENAI_API_KEY")))

from agents.research_agent.research_analyzer import _analyze_single_content, check_neutrality
from agents.research_agent.research_schemas import AnalyzedContent, ContentAnalysis, ScrapedContent

LangSmith key: True
OpenAI key: True


## 2. Load the saved report and images

In [3]:
DATA_DIR = project_root / "agents" / "research_agent" / "evaluation" / "data"
IMAGES_DIR = DATA_DIR / "images"
REPORT_PATH = DATA_DIR / "dearduck_sa_research_20260922_112749.json"

# The analyzer sends at most 10 images per post; the judges get the same ones.
MAX_IMAGES = 10
MIME_TYPES = {".jpg": "image/jpeg", ".jpeg": "image/jpeg", ".png": "image/png", ".webp": "image/webp"}

report = json.loads(REPORT_PATH.read_text(encoding="utf-8"))
print("Restaurant:", report["restaurant"]["name"])
print("Posts:", len(report["content"]))


def local_images(content_id: str) -> list[dict]:
    """Saved images for one post, in carousel order, as data URIs."""
    files = sorted(
        IMAGES_DIR.glob(f"{content_id}_*"),
        key=lambda path: int(path.stem.rsplit("_", 1)[1]),
    )[:MAX_IMAGES]
    images = []
    for path in files:
        mime_type = MIME_TYPES.get(path.suffix.lower(), "image/jpeg")
        encoded = base64.b64encode(path.read_bytes()).decode("utf-8")
        images.append({"mime_type": mime_type, "data": f"data:{mime_type};base64,{encoded}"})
    return images


missing = [item["content_id"] for item in report["content"] if not local_images(item["content_id"])]
print("Posts without saved images:", missing or "none")

Restaurant: Dear Duck
Posts: 20
Posts without saved images: none


## 3. LangSmith dataset

In [4]:
client = Client()
DATASET_NAME = "rawaj-research-content-analyzer-v2"

examples = [
    {
        "inputs": {
            "content": {
                "content_id": item["content_id"],
                "short_code": item["short_code"],
                "raw_data": item["raw_data"],
                "reel_details": item["reel_details"],
            }
        },
        "metadata": {
            "restaurant": report["restaurant"]["name"],
            "content_id": item["content_id"],
            "content_type": item["raw_data"]["content_type"],
            "is_reel": item["reel_details"] is not None,
        },
    }
    for item in report["content"]
]

# Safe to rerun: the dataset is only created the first time.
if client.has_dataset(dataset_name=DATASET_NAME):
    print("Dataset already exists:", DATASET_NAME)
else:
    dataset = client.create_dataset(
        dataset_name=DATASET_NAME,
        description="Instagram posts (text + saved images) for evaluating the Rawaj Research Agent content analyzer.",
    )
    client.create_examples(dataset_id=dataset.id, examples=examples)
    print("Dataset created:", DATASET_NAME, "with", len(examples), "examples")

Dataset created: rawaj-research-content-analyzer-v2 with 20 examples


## 4. Target: the content analyzer

In [5]:
def target(inputs: dict) -> dict:
    """Run the real analyzer on one post, using the saved images."""
    content = json.loads(json.dumps(inputs["content"]))  # copy
    content["raw_data"]["image_urls"] = [
        image["data"] for image in local_images(content["content_id"])
    ]
    analyzed = _analyze_single_content(content)
    # Only the analysis: the judges already get the post itself as input.
    return {"analysis": analyzed["analysis"]}

## 5. Evaluators

In [6]:
JUDGE_MODEL = "gpt-5.6-terra"

JUDGE_CONTEXT = """
You are evaluating Rawaj's Instagram Research Agent, which DESCRIBES one
restaurant Instagram post.

The INPUT is the original post evidence (caption, hashtags, metadata, Reel
transcript). The post's images are supplied below. The OUTPUT is the analysis
the Research Agent produced.

Do not require exact wording when two labels mean essentially the same thing.
Judge ONLY the criterion below; ignore problems that belong to other criteria.
"""

JUDGE_FOOTER = """
Return PASS (true) when the output is materially correct for this criterion.
Return FAIL (false) when there is a meaningful error for this criterion.
Briefly explain your decision, naming the exact field(s) at fault.

<input>
{inputs}
</input>

<output>
{outputs}
</output>

<images>
{attachments}
</images>
"""

CRITERIA = {
    "grounding": """
CRITERION: EVIDENCE GROUNDING AND FAITHFULNESS
- Every factual claim is supported by the caption, transcript, metadata or images.
- caption_summary, visual_summary and transcript_summary do not invent information.
- Each evidence item genuinely supports the fields it claims to support.
""",
    "visual_accuracy": """
CRITERION: VISUAL ACCURACY
Check against the supplied images:
- visual_signals: food_visible, menu_visible, price_visible, offer_visible,
  logo_visible, branding_visible, people_visible
- text_in_visual (text legible in the images)
- visual_summary
""",
    "interpretation": """
CRITERION: CONTENT INTERPRETATION
Check that these are reasonable readings of the evidence:
- content_pillar (the primary purpose of the post)
- content_categories and content_themes
- cta (present only if the audience is asked to do something)
- promotion (present only if something is pushed commercially: offer, discount,
  price, launch, event, giveaway; simply showing food is not a promotion)
""",
}

judges = {
    key: create_llm_as_judge(
        prompt=JUDGE_CONTEXT + criterion + JUDGE_FOOTER,
        model=JUDGE_MODEL,
        feedback_key=key,
    )
    for key, criterion in CRITERIA.items()
}


def judge_inputs(inputs: dict) -> dict:
    """The post as the judge sees it: expired image links replaced by a note."""
    content = json.loads(json.dumps(inputs["content"]))
    images = local_images(content["content_id"])
    content["raw_data"]["image_urls"] = f"{len(images)} image(s) supplied below"
    return {"content": content}


def make_llm_evaluator(key: str):
    def evaluator(inputs: dict, outputs: dict, reference_outputs=None):
        images = local_images(inputs["content"]["content_id"])
        if not images:
            raise ValueError(f"No saved images for {inputs['content']['content_id']}")
        return judges[key](inputs=judge_inputs(inputs), outputs=outputs, attachments=images)

    evaluator.__name__ = f"{key}_evaluator"
    return evaluator


def neutrality_evaluator(inputs: dict, outputs: dict, reference_outputs=None):
    """Code check (free): the analysis must describe, not judge."""
    item = AnalyzedContent(
        **ScrapedContent.model_validate(inputs["content"]).model_dump(),
        analysis=ContentAnalysis.model_validate(outputs["analysis"]),
    )
    warnings = check_neutrality([item])
    return {
        "key": "neutrality",
        "score": not warnings,
        "comment": "; ".join(warnings) or "No judgement words.",
    }


EVALUATORS = [*(make_llm_evaluator(key) for key in CRITERIA), neutrality_evaluator]

## 6. Try one post first (1 analyzer call + 3 judge calls)

In [7]:
test_input = examples[0]["inputs"]
test_output = target(test_input)
print(json.dumps(test_output, indent=2, ensure_ascii=False))

for evaluator in EVALUATORS:
    print(evaluator(inputs=test_input, outputs=test_output))

{
  "analysis": {
    "content_type": "Reel with promotional cover and music transcript",
    "content_categories": [
      "national_day_offer",
      "food_combo",
      "restaurant_booking"
    ],
    "content_pillar": "offers_campaigns",
    "content_themes": [
      "Saudi National Day",
      "Signature shakshouka-ish",
      "Chicken and Waffle",
      "96 SAR combo"
    ],
    "cta": {
      "present": true,
      "intent": "book a table",
      "description": "The audience is asked to book their spot today."
    },
    "promotion": {
      "present": true,
      "intent": "Saudi National Day combo offer",
      "description": "The post promotes Dear Duck Saudi National Day combos featuring Signature shakshouka-ish and Chicken and Waffle for 96 SAR."
    },
    "visual_signals": {
      "food_visible": true,
      "menu_visible": false,
      "price_visible": true,
      "offer_visible": true,
      "logo_visible": true,
      "branding_visible": true,
      "people_visible": f

## 7. Run the experiment

In [8]:
N_EXAMPLES = 20  
results = client.evaluate(
    target,
    data=client.list_examples(dataset_name=DATASET_NAME, limit=N_EXAMPLES),
    evaluators=EVALUATORS,
    experiment_prefix="rawaj-research-content-analyzer",
    description="Content analyzer: grounding, visual accuracy, interpretation (LLM judges) + neutrality (code).",
    max_concurrency=0,
)

c:\Users\asus\AppData\Local\Programs\Python\Python311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


View the evaluation results for experiment: 'rawaj-research-content-analyzer-ad1c97a9' at:
https://smith.langchain.com/o/49aa18a5-8e0b-4057-b558-c6df275d284d/datasets/14988184-cc3c-4320-9b3e-5f45a04bd662/compare?selectedSessions=085b5cc4-65e0-41c1-8e32-a30c49689fd4




20it [08:51, 26.58s/it]


## 8. Results

In [9]:
df = results.to_pandas()
score_columns = [column for column in df.columns if column.startswith("feedback.")]

print("Pass rate per evaluator:")
print((df[score_columns].astype(float).mean() * 100).round(1).astype(str) + "%")

# Posts that failed at least one evaluator, with the judges' explanations.
failed = df[(df[score_columns].astype(float) < 1).any(axis=1)]
print("\nPosts with at least one FAIL:", len(failed))

Pass rate per evaluator:
feedback.grounding           90.0%
feedback.visual_accuracy     80.0%
feedback.interpretation      85.0%
feedback.neutrality         100.0%
dtype: str

Posts with at least one FAIL: 8


In [10]:
# Why each post failed: the judge's explanation per FAIL
import textwrap

for _, row in df.iterrows():
    content = row["inputs.content"]
    for col in score_columns:
        if float(row[col]) < 1:
            key = col.removeprefix("feedback.")
            comment = row.get(f"feedback.{key}.comment") or row.get(f"feedback.comment") or ""
            print("=" * 80)
            print(f"POST {content['content_id']} | {content['raw_data']['content_type']} | FAIL: {key}")
            print("CAPTION:", content["raw_data"]["caption"][:150].replace("\n", " "))
            print("JUDGE:", textwrap.fill(str(comment), 100))


POST 3966482889807829390 | Video | FAIL: interpretation
CAPTION: Adult problems require adult solutions 😎  #dearduck #coffee #jeddah
JUDGE: 
POST 3970702584756402498 | Image | FAIL: visual_accuracy
CAPTION: When you don’t know where to go, go Full Brit 🍳  Served With Love, Everyday from 8AM-11PM💛🐣
JUDGE: 
POST 3962900373238397789 | Image | FAIL: visual_accuracy
CAPTION: Good things come in golden & crispy.   Don’t forget to stop by!🐣 #dearduck #alldayeatery #jeddah #fries
JUDGE: 
POST 3980253423001410785 | Sidecar | FAIL: interpretation
CAPTION: You can always count on her💛  Serving you daily, from 8AM-11PM☕️🐣
JUDGE: 
POST 3972280971397133293 | Video | FAIL: visual_accuracy
CAPTION: Stop by & Bring a friend 💛   Open Everyday, from 8AM-11PM🐣✨
JUDGE: 
POST 3969271939858791651 | Video | FAIL: grounding
CAPTION: New Ducks just landed and a new menu is coming sooooooon!! 🐣🥓🍳🍆🥫
JUDGE: 
POST 3969271939858791651 | Video | FAIL: visual_accuracy
CAPTION: New Ducks just landed and a new menu is c

In [11]:
for r in results:
    content = r["example"].inputs["content"]
    for er in r["evaluation_results"]["results"]:
        if er.score is not None and float(er.score) < 1:
            print("=" * 80)
            print(f"POST {content['content_id']} | FAIL: {er.key}")
            print("JUDGE:", er.comment)


POST 3966482889807829390 | FAIL: interpretation
JUDGE: The content categories, themes, entertainment/humor pillar, and absence of promotion are reasonable. However, the cta field is incorrectly marked present: “Who took the bomb?” is a line in the Reel transcript, not a clear audience-directed request or invitation to take an action. Thus, the score should be: false.
POST 3970702584756402498 | FAIL: visual_accuracy
JUDGE: FAIL. The `text_in_visual` field omits the prominent, legible headline: “SOME MORNINGS YOU HAVE TO GO ‘FULL BRIT’.” The `visual_summary` also says country loaf bread is shown, but no bread is visibly present on the photographed plate (it is only named in the visual text). The visual-signal flags are otherwise accurate. Thus, the score should be: false.
POST 3962900373238397789 | FAIL: visual_accuracy
JUDGE: FAIL. The fries visibly spell “Dear Duck,” which is legible brand text in the image. Therefore, `visual_signals.branding_visible` should be true and `text_in_visua